In dieser finalen Fusion werden die Feinstaubwerte der Sensor.Community und der Kopfklinik angehängt.
Hierzu wurden die Werte gemittelt, um synchrone Werte zu den Satellitenüberflugen zu haben und grobe Ausreißer rauszumitteln.
Die Zeitrange geht von 12:30 bis 14:30, hat also in etwa 40 Datenpunkte der Billigsensoren und zwei Datenpunkte der Kopfklinik.

In [1]:
import pandas as pd
import numpy as np

# =========================================================
# 1. Pfade definieren
# =========================================================
sensor_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\sensor_community__kopfklinik_may_august.csv"
master_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# Die finale Datei für das Random Forest Training
output_file = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv"

# =========================================================
# 2. Feinstaubdaten laden und zeitlich filtern
# =========================================================
print("Lade Feinstaub-Messdaten...")
df_sensors = pd.read_csv(sensor_file, sep=";", decimal=",")

# Zeitstempel ins Datetime-Format umwandeln
df_sensors['Zeitstempel'] = pd.to_datetime(df_sensors['Zeitstempel'], errors='coerce')

# PM2.5 in numerische Werte umwandeln
df_sensors['PM2.5'] = df_sensors['PM2.5'].astype(str).str.replace(",", ".", regex=False)
df_sensors['PM2.5'] = pd.to_numeric(df_sensors['PM2.5'], errors='coerce')

# Datum extrahieren für den späteren Merge
df_sensors['date'] = df_sensors['Zeitstempel'].dt.strftime('%Y-%m-%d')
df_sensors['station_id'] = df_sensors['sensor_id'].astype(str).str.strip()

# --- DER WICHTIGE ZEITFILTER (12:30 bis 14:30 Uhr) ---
print("Filtere auf das Satelliten-Überflugfenster (12:30 - 14:30 Uhr)...")
# Wir extrahieren die Uhrzeit
time_series = df_sensors['Zeitstempel'].dt.time
start_time = pd.to_datetime('12:30:00').time()
end_time = pd.to_datetime('14:30:00').time()

# Nur Daten behalten, die in dieses Fenster fallen
df_filtered = df_sensors[(time_series >= start_time) & (time_series <= end_time)].copy()

# =========================================================
# 3. Aggregation (Mittelwert pro Sensor und Tag)
# =========================================================
print("Berechne den mittäglichen PM2.5 Mittelwert pro Station und Tag...")
# Gruppieren nach Datum und Station, dann den Mittelwert von PM2.5 berechnen
daily_pm25 = df_filtered.groupby(['date', 'station_id'], as_index=False)['PM2.5'].mean()

# Spalte umbenennen, damit sie im ML-Modell klar als Zielvariable erkennbar ist
daily_pm25.rename(columns={'PM2.5': 'PM25_Target'}, inplace=True)

# Kurzer Check, wie viele gültige Tage/Stationen übrig bleiben
daily_pm25 = daily_pm25.dropna(subset=['PM25_Target'])
print(f"-> {len(daily_pm25)} gültige (Tag x Station) Kombinationen generiert.")

# =========================================================
# 4. Merge mit der Master-Feature-Datei
# =========================================================
print("Füge PM2.5 Target in die Feature-Masterdatei ein...")
df_features = pd.read_csv(master_file, sep=";", decimal=",")
df_features['station_id'] = df_features['station_id'].astype(str).str.strip()

# Den Left-Merge durchführen (Wir behalten alle Zeilen der Feature-Datei und hängen PM25_Target an)
df_final = pd.merge(
    df_features,
    daily_pm25,
    on=['date', 'station_id'],
    how='inner' # Inner Merge: Behält nur Tage, an denen wir Features UND Feinstaubmessungen haben!
)

# Optional: Nullen bei NO2 noch in NaN umwandeln (wie vorhin besprochen)
if 'S5P_NO2' in df_final.columns:
    df_final.loc[df_final['S5P_NO2'] == 0.0, 'S5P_NO2'] = np.nan

# =========================================================
# 5. Speichern
# =========================================================
df_final.to_csv(output_file, index=False, sep=";", decimal=",")

print("-" * 60)
print(f"MISSION ACCOMPLISHED! Der ML-Datensatz ist bereit für das Training:\n{output_file}")
print("-" * 60)

Lade Feinstaub-Messdaten...
Filtere auf das Satelliten-Überflugfenster (12:30 - 14:30 Uhr)...
Berechne den mittäglichen PM2.5 Mittelwert pro Station und Tag...
-> 850 gültige (Tag x Station) Kombinationen generiert.
Füge PM2.5 Target in die Feature-Masterdatei ein...
------------------------------------------------------------
MISSION ACCOMPLISHED! Der ML-Datensatz ist bereit für das Training:
C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\new_Output\IDW_hoch2\Exp2_Features_RH_PM25_NDVI_TPI_S5P_ERA5_MAIAC_Verkehr_Wetter.csv
------------------------------------------------------------
